In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [3]:
!pip install tf2onnx

  Using cached tf2onnx-1.17.0-py3-none-any.whl.metadata (27 kB)
  Using cached onnx-1.21.0-cp312-abi3-win_amd64.whl.metadata (8.7 kB)
Using cached tf2onnx-1.17.0-py3-none-any.whl (839 kB)
Using cached onnx-1.21.0-cp312-abi3-win_amd64.whl (16.4 MB)

   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------------- 0/2 [onnx]
   ---------------------------------

In [9]:
import tensorflow as tf

# Load Keras model
model = tf.keras.models.load_model("model_k.keras")

model.export("model_dir")
print("SavedFolder Sucessfully created!")

INFO:tensorflow:Assets written to: model_dir\assets


INFO:tensorflow:Assets written to: model_dir\assets


Saved artifact at 'model_dir'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 12), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  2871124572880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2871124572112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2871124573072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2871124571920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2871124573648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2871124571152: TensorSpec(shape=(), dtype=tf.resource, name=None)
SavedFolder Sucessfully created!


In [10]:
## load the encoder and scaler

with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [2]:
### Load the trained model, scaler pickle,onehot
model=load_model('model.h5')

## load the encoder and scaler
with open('onehot_encoder_geo.pkl','rb') as file:
    label_encoder_geo=pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [11]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

In [12]:
# One-hot encode 'Geography'
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df


c:\Users\biyan\Documents\python\myenv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [13]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [14]:
## Encode categorical variables
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,1,40,3,60000,2,1,1,50000


In [15]:
## concatination one hot encoded 
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [16]:
## Scaling the input data
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [17]:
## PRedict churn
prediction=model.predict(input_scaled)
prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step


array([[0.02236537]], dtype=float32)

In [18]:
prediction_proba = prediction[0][0]

In [19]:
prediction_proba

np.float32(0.022365374)

In [20]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
